In [34]:
from pathlib import Path
ROOT = Path("../..").resolve()

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (average_precision_score, fbeta_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt

df = pd.read_csv(ROOT / 'src/data/raw/bsNET140513_032310.csv')
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)


In [35]:
# Encode node IDs
all_nodes = pd.concat([df['Source'], df['Target']]).unique()
node2idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes = len(node2idx)

src = df['Source'].map(node2idx).values
tgt = df['Target'].map(node2idx).values
edge_index = torch.tensor(np.array([src, tgt]), dtype=torch.long)

In [36]:
n = len(df)
idx = np.random.permutation(n)
train_idx = idx[:int(0.8*n)]
val_idx   = idx[int(0.8*n):int(0.9*n)]
test_idx  = idx[int(0.9*n):]

In [37]:
from sklearn.preprocessing import StandardScaler

# Edge features
le = LabelEncoder()
type_enc = le.fit_transform(df['typeTrans'])  # 0-14

weight = df['Weight'].values.reshape(-1, 1)
scaler = StandardScaler()
# Scale weight, fit ONLY on train_idx
scaler.fit(weight[train_idx])
weight_scaled = scaler.transform(weight).flatten()

edge_attr = torch.tensor(
    np.stack([weight_scaled, type_enc], axis=1), dtype=torch.float
)

In [38]:
# Node features: degree + fraud rate per node (ONLY computed from train set)
node_feat = np.zeros((num_nodes, 2))

# In-degree & out-degree (computed for all edges, transductive structure)
node_feat[:, 0] += np.bincount(src, minlength=num_nodes)
node_feat[:, 0] += np.bincount(tgt, minlength=num_nodes)

# Fraud occurrences (strictly using training edges to prevent data leakage)
train_src = src[train_idx]
train_tgt = tgt[train_idx]
train_fraud = df['fraud'].values[train_idx]

pos_train_src = train_src[train_fraud == 1]
pos_train_tgt = train_tgt[train_fraud == 1]

node_feat[:, 1] += np.bincount(pos_train_src, minlength=num_nodes)
node_feat[:, 1] += np.bincount(pos_train_tgt, minlength=num_nodes)

In [39]:
# Normalize fraud count by degree
node_feat[:, 1] = node_feat[:, 1] / (node_feat[:, 0] + 1e-8)

# Normalize node features: fit on train nodes only
train_nodes = np.unique(np.concatenate([train_src, train_tgt]))
node_scaler = StandardScaler()
node_feat_scaled = node_scaler.fit(node_feat[train_nodes]).transform(node_feat)
x = torch.tensor(node_feat_scaled, dtype=torch.float)

# Labels (per edge)
y = torch.tensor(df['fraud'].values, dtype=torch.long)

# Verify normalization (~0 mean, ~1 std)
print(f"edge_attr[:,0]: mean={edge_attr[:,0].mean():.4f}, std={edge_attr[:,0].std():.4f}")
print(f"x[:,0]:         mean={x[:,0].mean():.4f}, std={x[:,0].std():.4f}")

edge_attr[:,0]: mean=0.0003, std=1.0176
x[:,0]:         mean=-0.0000, std=1.0001


In [40]:
class EdgeGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        # Edge classifier: concat src + tgt embeddings + edge_attr
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(hidden * 2 + 2, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(64, out_channels)
        )

    def forward(self, x, edge_index, edge_attr):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)

        # Edge representation
        src_emb = x[edge_index[0]]
        tgt_emb = x[edge_index[1]]
        edge_rep = torch.cat([src_emb, tgt_emb, edge_attr], dim=1)
        return self.fc(edge_rep).squeeze(-1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y).to(device)
model = EdgeGraphSAGE(in_channels=2, hidden=64, out_channels=1).to(device)

train_idx_t = torch.tensor(train_idx, dtype=torch.long).to(device)
val_idx_t   = torch.tensor(val_idx,   dtype=torch.long).to(device)
test_idx_t  = torch.tensor(test_idx,  dtype=torch.long).to(device)

# pos_weight calculated dynamically from train set
y_train = data.y[train_idx_t]
num_neg = (y_train == 0).sum().float()
num_pos = (y_train == 1).sum().float()
pos_weight = num_neg / (num_pos + 1e-8)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight.item()]).to(device)
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

train_losses, val_losses = [], []
val_prauc_history = []
best_val_prauc = 0
best_epoch = 0
patience, patience_counter = 20, 0

EPOCHS = 300

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(out[train_idx_t], data.y[train_idx_t].float())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    train_losses.append(loss.item())

    # --- Val ---
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index, data.edge_attr)
        val_loss = criterion(out[val_idx_t], data.y[val_idx_t].float()).item()
        val_losses.append(val_loss)

        probs = torch.sigmoid(out[val_idx_t]).cpu().numpy()
        labels = data.y[val_idx_t].cpu().numpy()
        prauc = average_precision_score(labels, probs)
        val_prauc_history.append(prauc)

    scheduler.step(prauc)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {loss.item():.4f} | "
              f"Val Loss: {val_loss:.4f} | Val PR-AUC: {prauc:.4f}")

    # Early stopping
    if prauc > best_val_prauc:
        best_val_prauc = prauc
        best_epoch = epoch
        torch.save(model.state_dict(), "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping tại epoch {epoch}")
            break

print(f"Best Val PR-AUC: {best_val_prauc:.4f} tại epoch {best_epoch}\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='blue')
axes[0].plot(val_losses,   label='Val Loss',   color='orange')
axes[0].axvline(best_epoch - 1, color='red', linestyle='--',
                label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].set_title('GraphSAGE - Train vs Val Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_prauc_history, label='Val PR-AUC', color='green')
axes[1].axvline(best_epoch - 1, color='red', linestyle='--',
                label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('GraphSAGE - Val PR-AUC theo Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('graphsage_loss_curve.png', dpi=150)
plt.show()


In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index, data.edge_attr)
    probs = torch.sigmoid(out[test_idx_t]).cpu().numpy()
    labels = data.y[test_idx_t].cpu().numpy()

# Optimal threshold via PR-AUC
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(labels, probs)

# Cắt bỏ phần tử cuối cùng để cùng chiều dài với thresholds
p_ = precision[:-1]
r_ = recall[:-1]
f2 = (5 * p_ * r_) / (4 * p_ + r_ + 1e-8)
best_thresh = thresholds[np.argmax(f2)]

preds = (probs >= best_thresh).astype(int)

print("\n=== TIER 2 TEST RESULTS ===")
print(f"PR-AUC:    {average_precision_score(labels, probs):.4f}")
print(f"F2-Score:  {fbeta_score(labels, preds, beta=2):.4f}")
print(f"Precision: {precision_score(labels, preds):.4f}")
print(f"Recall:    {recall_score(labels, preds):.4f}")
print(f"Threshold: {best_thresh:.4f}")
# Plot PR-AUC curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.', label=f'GraphSAGE (PR-AUC = {average_precision_score(labels, probs):.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Test Set)')
plt.legend()
plt.grid(True)
plt.show()
